# 1 Case Study: Classification with k-Nearest Neighbors and the Digits Dataset, Part 1


## 1.1 k-Nearest Neighbors Algorithm
### Hyperparameters and Hyperparameter Tuning

## 1.2 Loading the Dataset

In [ ]:
from sklearn.datasets import load_digits

In [ ]:
digits = load_digits()

### Displaying the Description

In [ ]:
print(digits.DESCR)

### Checking the Sample and Target Sizes

In [ ]:
digits.target[::100]

In [ ]:
digits.data.shape

In [ ]:
digits.target.shape

### A Sample Digit Image

In [ ]:
digits.images[13]

### Preparing the Data for Use with Scikit-Learn

In [ ]:
digits.data[13]

## 1.3 Visualizing the Data
### Creating the Diagram

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
figure, axes = plt.subplots(nrows=4, ncols=6, figsize=(6, 4))

### Displaying Each Image and Removing the Axes Labels

for item in zip(axes.ravel(), digits.images, digits.target):
    ax, image, target = item
    ax.imshow(image, cmap=plt.cm.gray)
    ax.set_xticks([])  # remove x-axis tick marks
    ax.set_yticks([])  # remove y-axis tick marks
    ax.set_title(target)
plt.tight_layout()

## 1.4 Splitting the Data for Training and Testing

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, random_state=11
)

### Training and Testing Set Sizes

In [ ]:
X_train.shape

In [ ]:
X_test.shape

## 1.5 Creating the Model

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

## 1.6 Training the Model

In [ ]:
knn.fit(X=X_train, y=y_train)

## 15.2.7 Predicting Digit Classes

In [ ]:
y_pred = knn.predict(X_test)

In [ ]:
y_pred[:20]

In [ ]:
y_test[:20]

In [ ]:
wrong = [(p.item(), e.item()) for (p, e) in zip(y_pred, y_test) if p != e]

In [ ]:
wrong

# 2 Case Study: Classification with k-Nearest Neighbors and the Digits Dataset, Part 2
## 2.1 Metrics for Model Accuracy
### Estimator Method `score`

In [ ]:
print(f"{knn.score(X_test, y_test):.2%}")

### Confusion Matrix

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_true=y_test,
    y_pred=y_pred,
)
plt.show()

In [ ]:
# Normalize because absolute numbers are not that nice
ConfusionMatrixDisplay.from_predictions(
    y_true=y_test,
    y_pred=y_pred,
    normalize="all",
)
plt.show()

### Classification Report

In [ ]:
from sklearn.metrics import classification_report

In [ ]:
print(classification_report(y_test, y_pred))

## 2.2 K-Fold Cross-Validation
### KFold Class

In [ ]:
from sklearn.model_selection import KFold

In [ ]:
kfold = KFold(n_splits=10, random_state=11, shuffle=True)

In [ ]:
kfold

### Using the `KFold` Object with Function `cross_val_score`

In [ ]:
from sklearn.model_selection import cross_val_score

In [ ]:
scores = cross_val_score(
    estimator=KNeighborsClassifier(n_neighbors=5),
    X=digits.data,
    y=digits.target,
    cv=kfold,
)

In [ ]:
print(scores)

In [ ]:
type(scores)

In [ ]:
print(f"Mean accuracy: {scores.mean():.2%}")

In [ ]:
print(f"Accuracy standard deviation: {scores.std():.2%}")

## 2.3 Running Multiple Models to Find the Best One

In [ ]:
from sklearn.svm import SVC

In [ ]:
from sklearn.naive_bayes import GaussianNB

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.neural_network import MLPClassifier

In [ ]:
estimators = {
    "KNeighborsClassifier": knn,
    "SVC": SVC(gamma="scale"),
    "SVC_lin": SVC(kernel="linear"),
    "GaussianNB": GaussianNB(),
    "LogisticRegression": LogisticRegression(solver="liblinear"),
    "NeuralNetwork": MLPClassifier(),
}

In [ ]:
for estimator_name, estimator_object in estimators.items():
    kfold = KFold(n_splits=10, random_state=11, shuffle=True)
    scores = cross_val_score(
        estimator=estimator_object, X=digits.data, y=digits.target, cv=kfold
    )
    print(
        f"{estimator_name:>20}: "
        f"mean accuracy={scores.mean():.2%}; "
        f"standard deviation={scores.std():.2%}"
    )

### Scikit-Learn Estimator Diagram

## 2.4 Hyperparameter Tuning

In [ ]:
avg_scores, std_scores = [], []
for k in range(1, 20 + 1):
    kfold = KFold(n_splits=7, random_state=11, shuffle=True)
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(estimator=knn, X=digits.data, y=digits.target, cv=kfold)
    mean, std = scores.mean(), scores.std()
    print(f"k={k:<2}; mean accuracy={mean:.2%}; standard deviation={std:.2%}")

    avg_scores.append(mean)
    std_scores.append(std)

In [ ]:
import numpy as np

fig, ax = plt.subplots()

ks = np.arange(1, len(avg_scores) + 1)

ax.errorbar(
    ks,
    avg_scores,
    std_scores,
    label="Mean Accuracy",
    fmt="o",
    capsize=6,
)

ax.set_xticks(ks)
ax.set_xlabel("$K$-NN")
ax.set_ylabel("Accuracy")

ax.set_ylim(0.97, 1.00)

ax.set_title("Number of Nearest Neighbor vs. Accuracy")
ax.legend()

plt.show()